# Chapitre 9 : construire des prompts complexes de A à Z

- [Cours](#cours)
- [Exercices](#exercices)
- [Espace de test](#espace-de-test)

## Configuration

Exécutez la cellule de configuration suivante pour charger votre clé API et définir la fonction utilitaire `get_completion`.

In [ ]:
!pip install anthropic

# Importer la bibliothèque standard de Python pour les expressions régulières
import re
import anthropic

# Récupérer les variables API_KEY et MODEL_NAME depuis le store IPython
%store -r API_KEY
%store -r MODEL_NAME

client = anthropic.Anthropic(api_key=API_KEY)

def get_completion(prompt: str, system_prompt="", prefill=""):
    message = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        system=system_prompt,
        messages=[
          {"role": "user", "content": prompt},
          {"role": "assistant", "content": prefill}
        ]
    )
    return message.content[0].text

---

## Cours

Félicitations, vous êtes arrivé au dernier chapitre ! Il est temps de tout assembler et d'apprendre à **créer des prompts complexes et sur mesure**.

Vous allez utiliser une **structure guidée que nous recommandons pour les prompts complexes**. Plus loin dans ce chapitre, nous présentons des prompts propres à certains secteurs et expliquons en quoi leur structure est similaire.

**Note :** **tous les prompts n'ont pas besoin de chaque élément de la structure complexe qui suit.** Nous vous encourageons à expérimenter, à ajouter ou retirer des éléments et à observer l'effet sur la réponse de Claude. Il est généralement **préférable d'utiliser beaucoup d'éléments pour obtenir d'abord un prompt qui fonctionne, puis de l'affiner et de l'alléger**.

### Exemple : chatbot coach de carrière

La structure suivante combine plusieurs éléments de prompt engineering et constitue un bon point de départ pour les prompts complexes. **L'ordre compte pour certains éléments**, pas pour d'autres. Nous signalons les cas où les bonnes pratiques imposent un ordre, mais en général, **respecter cet ordre est un bon début pour un excellent prompt**.

Dans l'exemple suivant, nous construisons un prompt pour un jeu de rôle encadré, dans lequel Claude endosse un rôle avec une tâche précise. L'objectif : que Claude se comporte comme un coach de carrière bienveillant.

Lisez puis exécutez la cellule ci-dessous pour assembler les différents éléments en un prompt complet.

In [ ]:
######################################## VARIABLES D'ENTRÉE ########################################

# Première variable d'entrée : l'historique de conversation (peut aussi être passé sous forme de messages `user` et `assistant` précédents dans l'appel API)
HISTORY = """Customer: Give me two possible careers for sociology majors.

Joe: Here are two potential careers for sociology majors:

- Social worker - Sociology provides a strong foundation for understanding human behavior and social systems. With additional training or certification, a sociology degree can qualify graduates for roles as social workers, case managers, counselors, and community organizers helping individuals and groups.

- Human resources specialist - An understanding of group dynamics and organizational behavior from sociology is applicable to careers in human resources. Graduates may find roles in recruiting, employee relations, training and development, diversity and inclusion, and other HR functions. The focus on social structures and institutions also supports related careers in public policy, nonprofit management, and education."""

# Deuxième variable d'entrée : la question de l'utilisateur
QUESTION = "Which of the two careers requires more than a Bachelor's degree?"



######################################## ÉLÉMENTS DU PROMPT ########################################

##### Élément 1 : rôle `user`
# Votre appel à l'API Messages doit toujours commencer par un rôle `user` dans le tableau messages.
# La fonction get_completion() définie plus haut le fait automatiquement pour vous.

##### Élément 2 : contexte de la tâche
# Donnez à Claude le contexte du rôle à endosser, ou des objectifs et de la tâche générale que le prompt doit lui faire accomplir.
# Il vaut mieux placer le contexte au début du prompt.
TASK_CONTEXT = "You will be acting as an AI career coach named Joe created by the company AdAstra Careers. Your goal is to give career advice to users. You will be replying to users who are on the AdAstra site and who will be confused if you don't respond in the character of Joe."

##### Élément 3 : ton
# Si c'est important pour l'échange, indiquez à Claude le ton à adopter.
# Selon la tâche, cet élément peut être inutile.
TONE_CONTEXT = "You should maintain a friendly customer service tone."

##### Élément 4 : description détaillée de la tâche et règles
# Détaillez les tâches précises que Claude doit réaliser et les règles qu'il doit respecter.
# C'est aussi ici que vous laissez à Claude une « porte de sortie » s'il n'a pas de réponse ou ne sait pas.
# Idéalement, faites relire cette description et ces règles à quelqu'un pour vérifier qu'elles sont logiques et que les termes ambigus sont bien définis.
TASK_DESCRIPTION = """Here are some important rules for the interaction:
- Always stay in character, as Joe, an AI from AdAstra Careers
- If you are unsure how to respond, say \"Sorry, I didn't understand that. Could you rephrase your question?\"
- If someone asks something irrelevant, say, \"Sorry, I am Joe and I give career advice. Do you have a career question today I can help you with?\""""

##### Élément 5 : exemples
# Fournissez à Claude au moins un exemple de réponse idéale à imiter, entre balises XML <example></example>. Vous pouvez en fournir plusieurs.
# Si vous fournissez plusieurs exemples, précisez à Claude ce que chacun illustre et placez chaque exemple dans ses propres balises XML.
# Les exemples sont sans doute l'outil le plus efficace pour obtenir de Claude le comportement voulu.
# Donnez des exemples des cas limites courants. Si votre prompt utilise un brouillon (scratchpad), montrez à quoi il doit ressembler.
# En général, plus il y a d'exemples, mieux c'est.
EXAMPLES = """Here is an example of how to respond in a standard interaction:
<example>
Customer: Hi, how were you created and what do you do?
Joe: Hello! My name is Joe, and I was created by AdAstra Careers to give career advice. What can I help you with today?
</example>"""

##### Élément 6 : données d'entrée à traiter
# Si Claude doit traiter des données dans le prompt, placez-les ici entre balises XML appropriées.
# Vous pouvez inclure plusieurs données, chacune dans ses propres balises XML.
# Selon la tâche, cet élément peut être inutile. Son emplacement est libre.
INPUT_DATA = f"""Here is the conversational history (between the user and you) prior to the question. It could be empty if there is no history:
<history>
{HISTORY}
</history>

Here is the user's question:
<question>
{QUESTION}
</question>"""

##### Élément 7 : description ou demande de la tâche immédiate #####
# « Rappelez » à Claude, ou dites-lui précisément, ce qu'il doit faire immédiatement pour accomplir la tâche.
# C'est aussi ici que vous placez des variables supplémentaires, comme la question de l'utilisateur.
# Répéter la tâche immédiate ne nuit généralement pas. Faites-le plutôt vers la fin d'un long prompt.
# Les résultats seront meilleurs qu'en la plaçant au début.
# Il est aussi recommandé de placer la question de l'utilisateur vers la fin du prompt.
IMMEDIATE_TASK = "How do you respond to the user's question?"

##### Élément 8 : réflexion préalable (raisonnement pas à pas)
# Pour les tâches en plusieurs étapes, demandez à Claude de réfléchir étape par étape avant de répondre.
# Il faut parfois écrire « Avant de donner ta réponse... » pour que Claude le fasse bien en premier.
# Pas nécessaire pour tous les prompts. Si vous l'incluez, placez-le vers la fin d'un long prompt, juste après la demande de tâche immédiate.
PRECOGNITION = "Think about your answer first before you respond."

##### Élément 9 : format de sortie
# Si la réponse doit suivre un format précis, décrivez clairement ce format à Claude.
# Selon la tâche, cet élément peut être inutile.
# Si vous l'incluez, placez-le plutôt vers la fin du prompt qu'au début.
OUTPUT_FORMATTING = "Put your response in <response></response> tags."

##### Élément 10 : pré-remplissage de la réponse de Claude (facultatif)
# Emplacement pour commencer la réponse de Claude par quelques mots afin d'orienter son comportement ou sa réponse.
# Pour pré-remplir la réponse de Claude, placez ce texte dans le rôle `assistant` de l'appel API.
# Selon la tâche, cet élément peut être inutile.
PREFILL = "[Joe] <response>"



######################################## ASSEMBLAGE DES ÉLÉMENTS ########################################

PROMPT = ""

if TASK_CONTEXT:
    PROMPT += f"""{TASK_CONTEXT}"""

if TONE_CONTEXT:
    PROMPT += f"""\n\n{TONE_CONTEXT}"""

if TASK_DESCRIPTION:
    PROMPT += f"""\n\n{TASK_DESCRIPTION}"""

if EXAMPLES:
    PROMPT += f"""\n\n{EXAMPLES}"""

if INPUT_DATA:
    PROMPT += f"""\n\n{INPUT_DATA}"""

if IMMEDIATE_TASK:
    PROMPT += f"""\n\n{IMMEDIATE_TASK}"""

if PRECOGNITION:
    PROMPT += f"""\n\n{PRECOGNITION}"""

if OUTPUT_FORMATTING:
    PROMPT += f"""\n\n{OUTPUT_FORMATTING}"""

# Afficher le prompt complet
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN")
print(PROMPT)
print("\nASSISTANT TURN")
print(PREFILL)

Exécutons maintenant le prompt ! Lancez la cellule ci-dessous pour voir la réponse de Claude.

In [ ]:
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))

### Exemple : services juridiques

**Les prompts du domaine juridique peuvent être assez complexes**, car il faut :
- analyser de longs documents ;
- traiter des sujets complexes ;
- formater la sortie de façon très précise ;
- suivre des processus d'analyse en plusieurs étapes.

Voyons comment utiliser le modèle de prompt complexe pour structurer un cas d'usage juridique précis. Ci-dessous, nous détaillons un exemple de prompt dans lequel Claude répond à des questions sur un problème juridique à partir des informations d'un document juridique.

Nous avons **modifié l'ordre de quelques éléments** pour montrer que la structure d'un prompt peut être souple.

**Le prompt engineering procède par essais et erreurs, de façon méthodique.** Nous vous encourageons à combiner, à déplacer les éléments (ceux dont l'ordre n'a pas d'importance) et à voir ce qui fonctionne le mieux pour vos besoins.

In [ ]:
######################################## VARIABLES D'ENTRÉE ########################################

# Première variable d'entrée : le document juridique
LEGAL_RESEARCH = """<search_results>
<search_result id=1>
The animal health industry became caught up in a number of patent and trademark lawsuits during the past year. In 1994, Barclay Slocum obtained patents for the tibial plateau leveling osteotomy procedure, which is used in the treatment of dogs with cranial cruciate ligament rupture, and for the devices used in the procedure. During 2006, Slocum Enterprises filed a patent infringement suit against New Generation Devices, arguing that the Unity Cruciate Plate manufactured by New Generation infringed on the patent for the Slocum TPLO plate. However, the court never reached a decision on the issue of patent infringement, ruling that it did not have jurisdiction on the basis of the small number of plates sold in the state in which the case was filed and the information provided on a Web site maintained by Slocum Enterprises. Other patent battles waged during 2006 concerned the use of laser technology for onychectomy in cats, pet identification chips, pig vaccines, and pet “deshedding” tools.
</search_result>
<search_result id=2>
In Canada, the British Columbia Veterinary Medical Association brought suit against a nonveterinarian, claiming that he engaged in cutting or otherwise removing hooks from horses' teeth and floating horses' teeth with power and manual tools, provided advice and diagnoses in return for a fee, and held himself out as being qualified and willing to provide treatment with respect to these activities. The court held that the intention of the legislature in passing the Veterinary Profession Act was the protection of the public and animals and further held that monopolistic statutes serve the purpose of protecting the public. In addition, the court concluded that dentistry, at its core, relates to the health of the teeth and gums; is distinct from cosmetic and other types of care of animals; and, therefore, falls under the definition of the practice of veterinary medicine. The nonveterinarian was enjoined from providing services without a veterinarian supervising the procedures.
</search_result>
<search_result id=3>
The aftermath of Hurricane Katrina, which hit the Gulf Coast of the United States during 2005, spurred changes to the way animals are treated during natural disasters. In 2006, Hawaii, Louisiana, and New Hampshire all enacted laws that address issues regarding the care of animals during disasters, such as providing shelters for pets and allowing service animals to be kept with the people they serve. In addition, Congress passed, and the President signed, the Pet Evacuation and Transportation Standards Act during 2006, which requires state and local emergency preparedness authorities to include in their evacuation plans information on how they will accommodate household pets and service animals in case of a disaster. California passed a law that will require its Office of Emergency Services, Department of Agriculture, and other agencies involved with disaster response preparation to develop a plan for the needs of service animals, livestock, equids, and household pets in the event of a disaster or major emergency.
</search_result>
</search_results>"""

# Deuxième variable d'entrée : la question de l'utilisateur
QUESTION = "Are there any laws about what to do with pets during a hurricane?"



######################################## ÉLÉMENTS DU PROMPT ########################################

##### Élément 1 : rôle `user`
# Votre appel à l'API Messages doit toujours commencer par un rôle `user` dans le tableau messages.
# La fonction get_completion() définie plus haut le fait automatiquement pour vous.

##### Élément 2 : contexte de la tâche
# Donnez à Claude le contexte du rôle à endosser, ou des objectifs et de la tâche générale que le prompt doit lui faire accomplir.
# Il vaut mieux placer le contexte au début du prompt.
TASK_CONTEXT = "You are an expert lawyer."

##### Élément 3 : ton
# Si c'est important pour l'échange, indiquez à Claude le ton à adopter.
# Selon la tâche, cet élément peut être inutile.
TONE_CONTEXT = ""

##### Élément 4 : données d'entrée à traiter
# Si Claude doit traiter des données dans le prompt, placez-les ici entre balises XML appropriées.
# Vous pouvez inclure plusieurs données, chacune dans ses propres balises XML.
# Selon la tâche, cet élément peut être inutile. Son emplacement est libre.
INPUT_DATA = f"""Here is some research that's been compiled. Use it to answer a legal question from the user.
<legal_research>
{LEGAL_RESEARCH}
</legal_research>"""

##### Élément 5 : exemples
# Fournissez à Claude au moins un exemple de réponse idéale à imiter, entre balises XML <example></example>. Vous pouvez en fournir plusieurs.
# Si vous fournissez plusieurs exemples, précisez à Claude ce que chacun illustre et placez chaque exemple dans ses propres balises XML.
# Les exemples sont sans doute l'outil le plus efficace pour obtenir de Claude le comportement voulu.
# Donnez des exemples des cas limites courants. Si votre prompt utilise un brouillon (scratchpad), montrez à quoi il doit ressembler.
# En général, plus il y a d'exemples, mieux c'est.
EXAMPLES = """When citing the legal research in your answer, please use brackets containing the search index ID, followed by a period. Put these at the end of the sentence that's doing the citing. Examples of proper citation format:

<examples>
<example>
The statute of limitations expires after 10 years for crimes like this. [3].
</example>
<example>
However, the protection does not apply when it has been specifically waived by both parties. [5].
</example>
</examples>"""

##### Élément 6 : description détaillée de la tâche et règles
# Détaillez les tâches précises que Claude doit réaliser et les règles qu'il doit respecter.
# C'est aussi ici que vous laissez à Claude une « porte de sortie » s'il n'a pas de réponse ou ne sait pas.
# Idéalement, faites relire cette description et ces règles à quelqu'un pour vérifier qu'elles sont logiques et que les termes ambigus sont bien définis.
TASK_DESCRIPTION = """Write a clear, concise answer to this question:

<question>
{QUESTION}
</question>

It should be no more than a couple of paragraphs. If possible, it should conclude with a single sentence directly answering the user's question. However, if there is not sufficient information in the compiled research to produce such an answer, you may demur and write "Sorry, I do not have sufficient information at hand to answer this question."."""

##### Élément 7 : description ou demande de la tâche immédiate #####
# « Rappelez » à Claude, ou dites-lui précisément, ce qu'il doit faire immédiatement pour accomplir la tâche.
# C'est aussi ici que vous placez des variables supplémentaires, comme la question de l'utilisateur.
# Répéter la tâche immédiate ne nuit généralement pas. Faites-le plutôt vers la fin d'un long prompt.
# Les résultats seront meilleurs qu'en la plaçant au début.
# Il est aussi recommandé de placer la question de l'utilisateur vers la fin du prompt.
IMMEDIATE_TASK = ""

##### Élément 8 : réflexion préalable (raisonnement pas à pas)
# Pour les tâches en plusieurs étapes, demandez à Claude de réfléchir étape par étape avant de répondre.
# Il faut parfois écrire « Avant de donner ta réponse... » pour que Claude le fasse bien en premier.
# Pas nécessaire pour tous les prompts. Si vous l'incluez, placez-le vers la fin d'un long prompt, juste après la demande de tâche immédiate.
PRECOGNITION = "Before you answer, pull out the most relevant quotes from the research in <relevant_quotes> tags."

##### Élément 9 : format de sortie
# Si la réponse doit suivre un format précis, décrivez clairement ce format à Claude.
# Selon la tâche, cet élément peut être inutile.
# Si vous l'incluez, placez-le plutôt vers la fin du prompt qu'au début.
OUTPUT_FORMATTING = "Put your two-paragraph response in <answer> tags."

##### Élément 10 : pré-remplissage de la réponse de Claude (facultatif)
# Emplacement pour commencer la réponse de Claude par quelques mots afin d'orienter son comportement ou sa réponse.
# Pour pré-remplir la réponse de Claude, placez ce texte dans le rôle `assistant` de l'appel API.
# Selon la tâche, cet élément peut être inutile.
PREFILL = "<relevant_quotes>"



######################################## ASSEMBLAGE DES ÉLÉMENTS ########################################

PROMPT = ""

if TASK_CONTEXT:
    PROMPT += f"""{TASK_CONTEXT}"""

if TONE_CONTEXT:
    PROMPT += f"""\n\n{TONE_CONTEXT}"""

if INPUT_DATA:
    PROMPT += f"""\n\n{INPUT_DATA}"""

if EXAMPLES:
    PROMPT += f"""\n\n{EXAMPLES}"""

if TASK_DESCRIPTION:
    PROMPT += f"""\n\n{TASK_DESCRIPTION}"""

if IMMEDIATE_TASK:
    PROMPT += f"""\n\n{IMMEDIATE_TASK}"""

if PRECOGNITION:
    PROMPT += f"""\n\n{PRECOGNITION}"""

if OUTPUT_FORMATTING:
    PROMPT += f"""\n\n{OUTPUT_FORMATTING}"""

# Afficher le prompt complet
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN")
print(PROMPT)
print("\nASSISTANT TURN")
print(PREFILL)

Exécutons maintenant le prompt ! Lancez la cellule ci-dessous pour voir la réponse de Claude.

In [ ]:
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))

Si vous voulez expérimenter avec les prompts du cours sans modifier le contenu ci-dessus, descendez tout en bas du notebook jusqu'à l'[**Espace de test**](#espace-de-test).

---

## Exercices
- [Exercice 9.1 - Chatbot de services financiers](#exercice-91---chatbot-de-services-financiers)
- [Exercice 9.2 - Codebot](#exercice-92---codebot)

### Exercice 9.1 - Chatbot de services financiers
Les prompts du domaine financier peuvent aussi être complexes, pour des raisons proches de celles du juridique. Voici un exercice de cas d'usage financier, dans lequel Claude sert à **analyser des informations fiscales et répondre à des questions**. Comme pour l'exemple juridique, nous avons modifié l'ordre de quelques éléments, car notre solution est plus logique avec un autre enchaînement (d'autres structures fonctionneraient aussi).

Nous vous conseillons de lire le contenu des variables (ici `{QUESTION}` et `{TAX_CODE}`) pour comprendre sur quoi Claude doit travailler. Veillez à faire référence directement à `{QUESTION}` et `{TAX_CODE}` quelque part dans votre prompt (avec la syntaxe f-string, comme dans les autres exemples) pour que le contenu réel des variables y soit inséré.

Remplissez les champs des éléments du prompt avec un contenu conforme à leur description et aux exemples de prompts complexes vus plus haut. Une fois les éléments voulus remplis, exécutez la cellule pour voir le prompt assemblé et la réponse de Claude.

Rappelez-vous que le prompt engineering est rarement une simple formule, surtout pour les prompts longs et complexes ! Il est important de construire des cas de test et **d'essayer plusieurs prompts et structures pour voir ce qui fonctionne le mieux dans chaque situation**. Si vous modifiez l'ordre des éléments, pensez à modifier aussi l'ordre de concaténation dans la section `COMBINE ELEMENTS`.

In [ ]:
######################################## VARIABLES D'ENTRÉE ########################################

# Première variable d'entrée : la question de l'utilisateur
QUESTION = "How long do I have to make an 83b election?"

# Deuxième variable d'entrée : le code fiscal que Claude utilisera pour répondre
TAX_CODE = """
(a)General rule
If, in connection with the performance of services, property is transferred to any person other than the person for whom such services are performed, the excess of—
(1)the fair market value of such property (determined without regard to any restriction other than a restriction which by its terms will never lapse) at the first time the rights of the person having the beneficial interest in such property are transferable or are not subject to a substantial risk of forfeiture, whichever occurs earlier, over
(2)the amount (if any) paid for such property,
shall be included in the gross income of the person who performed such services in the first taxable year in which the rights of the person having the beneficial interest in such property are transferable or are not subject to a substantial risk of forfeiture, whichever is applicable. The preceding sentence shall not apply if such person sells or otherwise disposes of such property in an arm’s length transaction before his rights in such property become transferable or not subject to a substantial risk of forfeiture.
(b)Election to include in gross income in year of transfer
(1)In general
Any person who performs services in connection with which property is transferred to any person may elect to include in his gross income for the taxable year in which such property is transferred, the excess of—
(A)the fair market value of such property at the time of transfer (determined without regard to any restriction other than a restriction which by its terms will never lapse), over
(B)the amount (if any) paid for such property.
If such election is made, subsection (a) shall not apply with respect to the transfer of such property, and if such property is subsequently forfeited, no deduction shall be allowed in respect of such forfeiture.
(2)Election
An election under paragraph (1) with respect to any transfer of property shall be made in such manner as the Secretary prescribes and shall be made not later than 30 days after the date of such transfer. Such election may not be revoked except with the consent of the Secretary.

(c)Special rules
For purposes of this section—
(1)Substantial risk of forfeiture
The rights of a person in property are subject to a substantial risk of forfeiture if such person’s rights to full enjoyment of such property are conditioned upon the future performance of substantial services by any individual.

(2)Transferability of property
The rights of a person in property are transferable only if the rights in such property of any transferee are not subject to a substantial risk of forfeiture.

(3)Sales which may give rise to suit under section 16(b) of the Securities Exchange Act of 1934
So long as the sale of property at a profit could subject a person to suit under section 16(b) of the Securities Exchange Act of 1934, such person’s rights in such property are—
(A)subject to a substantial risk of forfeiture, and
(B)not transferable.
(4)For purposes of determining an individual’s basis in property transferred in connection with the performance of services, rules similar to the rules of section 72(w) shall apply.
(d)Certain restrictions which will never lapse
(1)Valuation
In the case of property subject to a restriction which by its terms will never lapse, and which allows the transferee to sell such property only at a price determined under a formula, the price so determined shall be deemed to be the fair market value of the property unless established to the contrary by the Secretary, and the burden of proof shall be on the Secretary with respect to such value.

(2)Cancellation
If, in the case of property subject to a restriction which by its terms will never lapse, the restriction is canceled, then, unless the taxpayer establishes—
(A)that such cancellation was not compensatory, and
(B)that the person, if any, who would be allowed a deduction if the cancellation were treated as compensatory, will treat the transaction as not compensatory, as evidenced in such manner as the Secretary shall prescribe by regulations,
the excess of the fair market value of the property (computed without regard to the restrictions) at the time of cancellation over the sum of—
(C)the fair market value of such property (computed by taking the restriction into account) immediately before the cancellation, and
(D)the amount, if any, paid for the cancellation,
shall be treated as compensation for the taxable year in which such cancellation occurs.
(e)Applicability of section
This section shall not apply to—
(1)a transaction to which section 421 applies,
(2)a transfer to or from a trust described in section 401(a) or a transfer under an annuity plan which meets the requirements of section 404(a)(2),
(3)the transfer of an option without a readily ascertainable fair market value,
(4)the transfer of property pursuant to the exercise of an option with a readily ascertainable fair market value at the date of grant, or
(5)group-term life insurance to which section 79 applies.
(f)Holding period
In determining the period for which the taxpayer has held property to which subsection (a) applies, there shall be included only the period beginning at the first time his rights in such property are transferable or are not subject to a substantial risk of forfeiture, whichever occurs earlier.

(g)Certain exchanges
If property to which subsection (a) applies is exchanged for property subject to restrictions and conditions substantially similar to those to which the property given in such exchange was subject, and if section 354, 355, 356, or 1036 (or so much of section 1031 as relates to section 1036) applied to such exchange, or if such exchange was pursuant to the exercise of a conversion privilege—
(1)such exchange shall be disregarded for purposes of subsection (a), and
(2)the property received shall be treated as property to which subsection (a) applies.
(h)Deduction by employer
In the case of a transfer of property to which this section applies or a cancellation of a restriction described in subsection (d), there shall be allowed as a deduction under section 162, to the person for whom were performed the services in connection with which such property was transferred, an amount equal to the amount included under subsection (a), (b), or (d)(2) in the gross income of the person who performed such services. Such deduction shall be allowed for the taxable year of such person in which or with which ends the taxable year in which such amount is included in the gross income of the person who performed such services.

(i)Qualified equity grants
(1)In general
For purposes of this subtitle—
(A)Timing of inclusion
If qualified stock is transferred to a qualified employee who makes an election with respect to such stock under this subsection, subsection (a) shall be applied by including the amount determined under such subsection with respect to such stock in income of the employee in the taxable year determined under subparagraph (B) in lieu of the taxable year described in subsection (a).

(B)Taxable year determined
The taxable year determined under this subparagraph is the taxable year of the employee which includes the earliest of—
(i)the first date such qualified stock becomes transferable (including, solely for purposes of this clause, becoming transferable to the employer),
(ii)the date the employee first becomes an excluded employee,
(iii)the first date on which any stock of the corporation which issued the qualified stock becomes readily tradable on an established securities market (as determined by the Secretary, but not including any market unless such market is recognized as an established securities market by the Secretary for purposes of a provision of this title other than this subsection),
(iv)the date that is 5 years after the first date the rights of the employee in such stock are transferable or are not subject to a substantial risk of forfeiture, whichever occurs earlier, or
(v)the date on which the employee revokes (at such time and in such manner as the Secretary provides) the election under this subsection with respect to such stock.
(2)Qualified stock
(A)In general
For purposes of this subsection, the term “qualified stock” means, with respect to any qualified employee, any stock in a corporation which is the employer of such employee, if—
(i)such stock is received—
(I)in connection with the exercise of an option, or
(II)in settlement of a restricted stock unit, and
(ii)such option or restricted stock unit was granted by the corporation—
(I)in connection with the performance of services as an employee, and
(II)during a calendar year in which such corporation was an eligible corporation.
(B)Limitation
The term “qualified stock” shall not include any stock if the employee may sell such stock to, or otherwise receive cash in lieu of stock from, the corporation at the time that the rights of the employee in such stock first become transferable or not subject to a substantial risk of forfeiture.

(C)Eligible corporation
For purposes of subparagraph (A)(ii)(II)—
(i)In general
The term “eligible corporation” means, with respect to any calendar year, any corporation if—
(I)no stock of such corporation (or any predecessor of such corporation) is readily tradable on an established securities market (as determined under paragraph (1)(B)(iii)) during any preceding calendar year, and
(II)such corporation has a written plan under which, in such calendar year, not less than 80 percent of all employees who provide services to such corporation in the United States (or any possession of the United States) are granted stock options, or are granted restricted stock units, with the same rights and privileges to receive qualified stock.
(ii)Same rights and privileges
For purposes of clause (i)(II)—
(I)except as provided in subclauses (II) and (III), the determination of rights and privileges with respect to stock shall be made in a similar manner as under section 423(b)(5),
(II)employees shall not fail to be treated as having the same rights and privileges to receive qualified stock solely because the number of shares available to all employees is not equal in amount, so long as the number of shares available to each employee is more than a de minimis amount, and
(III)rights and privileges with respect to the exercise of an option shall not be treated as the same as rights and privileges with respect to the settlement of a restricted stock unit.
(iii)Employee
For purposes of clause (i)(II), the term “employee” shall not include any employee described in section 4980E(d)(4) or any excluded employee.

(iv)Special rule for calendar years before 2018
In the case of any calendar year beginning before January 1, 2018, clause (i)(II) shall be applied without regard to whether the rights and privileges with respect to the qualified stock are the same.

(3)Qualified employee; excluded employee
For purposes of this subsection—
(A)In general
The term “qualified employee” means any individual who—
(i)is not an excluded employee, and
(ii)agrees in the election made under this subsection to meet such requirements as are determined by the Secretary to be necessary to ensure that the withholding requirements of the corporation under chapter 24 with respect to the qualified stock are met.
(B)Excluded employee
The term “excluded employee” means, with respect to any corporation, any individual—
(i)who is a 1-percent owner (within the meaning of section 416(i)(1)(B)(ii)) at any time during the calendar year or who was such a 1 percent owner at any time during the 10 preceding calendar years,
(ii)who is or has been at any prior time—
(I)the chief executive officer of such corporation or an individual acting in such a capacity, or
(II)the chief financial officer of such corporation or an individual acting in such a capacity,
(iii)who bears a relationship described in section 318(a)(1) to any individual described in subclause (I) or (II) of clause (ii), or
(iv)who is one of the 4 highest compensated officers of such corporation for the taxable year, or was one of the 4 highest compensated officers of such corporation for any of the 10 preceding taxable years, determined with respect to each such taxable year on the basis of the shareholder disclosure rules for compensation under the Securities Exchange Act of 1934 (as if such rules applied to such corporation).
(4)Election
(A)Time for making election
An election with respect to qualified stock shall be made under this subsection no later than 30 days after the first date the rights of the employee in such stock are transferable or are not subject to a substantial risk of forfeiture, whichever occurs earlier, and shall be made in a manner similar to the manner in which an election is made under subsection (b).

(B)Limitations
No election may be made under this section with respect to any qualified stock if—
(i)the qualified employee has made an election under subsection (b) with respect to such qualified stock,
(ii)any stock of the corporation which issued the qualified stock is readily tradable on an established securities market (as determined under paragraph (1)(B)(iii)) at any time before the election is made, or
(iii)such corporation purchased any of its outstanding stock in the calendar year preceding the calendar year which includes the first date the rights of the employee in such stock are transferable or are not subject to a substantial risk of forfeiture, unless—
(I)not less than 25 percent of the total dollar amount of the stock so purchased is deferral stock, and
(II)the determination of which individuals from whom deferral stock is purchased is made on a reasonable basis.
(C)Definitions and special rules related to limitation on stock redemptions
(i)Deferral stock
For purposes of this paragraph, the term “deferral stock” means stock with respect to which an election is in effect under this subsection.

(ii)Deferral stock with respect to any individual not taken into account if individual holds deferral stock with longer deferral period
Stock purchased by a corporation from any individual shall not be treated as deferral stock for purposes of subparagraph (B)(iii) if such individual (immediately after such purchase) holds any deferral stock with respect to which an election has been in effect under this subsection for a longer period than the election with respect to the stock so purchased.

(iii)Purchase of all outstanding deferral stock
The requirements of subclauses (I) and (II) of subparagraph (B)(iii) shall be treated as met if the stock so purchased includes all of the corporation’s outstanding deferral stock.

(iv)Reporting
Any corporation which has outstanding deferral stock as of the beginning of any calendar year and which purchases any of its outstanding stock during such calendar year shall include on its return of tax for the taxable year in which, or with which, such calendar year ends the total dollar amount of its outstanding stock so purchased during such calendar year and such other information as the Secretary requires for purposes of administering this paragraph.

(5)Controlled groups
For purposes of this subsection, all persons treated as a single employer under section 414(b) shall be treated as 1 corporation.

(6)Notice requirement
Any corporation which transfers qualified stock to a qualified employee shall, at the time that (or a reasonable period before) an amount attributable to such stock would (but for this subsection) first be includible in the gross income of such employee—
(A)certify to such employee that such stock is qualified stock, and
(B)notify such employee—
(i)that the employee may be eligible to elect to defer income on such stock under this subsection, and
(ii)that, if the employee makes such an election—
(I)the amount of income recognized at the end of the deferral period will be based on the value of the stock at the time at which the rights of the employee in such stock first become transferable or not subject to substantial risk of forfeiture, notwithstanding whether the value of the stock has declined during the deferral period,
(II)the amount of such income recognized at the end of the deferral period will be subject to withholding under section 3401(i) at the rate determined under section 3402(t), and
(III)the responsibilities of the employee (as determined by the Secretary under paragraph (3)(A)(ii)) with respect to such withholding.
(7)Restricted stock units
This section (other than this subsection), including any election under subsection (b), shall not apply to restricted stock units.
"""



######################################## ÉLÉMENTS DU PROMPT ########################################

##### Élément 1 : rôle `user`
# Votre appel à l'API Messages doit toujours commencer par un rôle `user` dans le tableau messages.
# La fonction get_completion() définie plus haut le fait automatiquement pour vous.

##### Élément 2 : contexte de la tâche
# Donnez à Claude le contexte du rôle à endosser, ou des objectifs et de la tâche générale que le prompt doit lui faire accomplir.
# Il vaut mieux placer le contexte au début du prompt.
TASK_CONTEXT = ""

##### Élément 3 : ton
# Si c'est important pour l'échange, indiquez à Claude le ton à adopter.
# Selon la tâche, cet élément peut être inutile.
TONE_CONTEXT = ""

##### Élément 4 : données d'entrée à traiter
# Si Claude doit traiter des données dans le prompt, placez-les ici entre balises XML appropriées.
# Vous pouvez inclure plusieurs données, chacune dans ses propres balises XML.
# Selon la tâche, cet élément peut être inutile. Son emplacement est libre.
INPUT_DATA = ""

##### Élément 5 : exemples
# Fournissez à Claude au moins un exemple de réponse idéale à imiter, entre balises XML <example></example>. Vous pouvez en fournir plusieurs.
# Si vous fournissez plusieurs exemples, précisez à Claude ce que chacun illustre et placez chaque exemple dans ses propres balises XML.
# Les exemples sont sans doute l'outil le plus efficace pour obtenir de Claude le comportement voulu.
# Donnez des exemples des cas limites courants. Si votre prompt utilise un brouillon (scratchpad), montrez à quoi il doit ressembler.
# En général, plus il y a d'exemples, mieux c'est.
EXAMPLES = ""

##### Élément 6 : description détaillée de la tâche et règles
# Détaillez les tâches précises que Claude doit réaliser et les règles qu'il doit respecter.
# C'est aussi ici que vous laissez à Claude une « porte de sortie » s'il n'a pas de réponse ou ne sait pas.
# Idéalement, faites relire cette description et ces règles à quelqu'un pour vérifier qu'elles sont logiques et que les termes ambigus sont bien définis.
TASK_DESCRIPTION = ""

##### Élément 7 : description ou demande de la tâche immédiate #####
# « Rappelez » à Claude, ou dites-lui précisément, ce qu'il doit faire immédiatement pour accomplir la tâche.
# C'est aussi ici que vous placez des variables supplémentaires, comme la question de l'utilisateur.
# Répéter la tâche immédiate ne nuit généralement pas. Faites-le plutôt vers la fin d'un long prompt.
# Les résultats seront meilleurs qu'en la plaçant au début.
# Il est aussi recommandé de placer la question de l'utilisateur vers la fin du prompt.
IMMEDIATE_TASK = ""

##### Élément 8 : réflexion préalable (raisonnement pas à pas)
# Pour les tâches en plusieurs étapes, demandez à Claude de réfléchir étape par étape avant de répondre.
# Il faut parfois écrire « Avant de donner ta réponse... » pour que Claude le fasse bien en premier.
# Pas nécessaire pour tous les prompts. Si vous l'incluez, placez-le vers la fin d'un long prompt, juste après la demande de tâche immédiate.
PRECOGNITION = ""

##### Élément 9 : format de sortie
# Si la réponse doit suivre un format précis, décrivez clairement ce format à Claude.
# Selon la tâche, cet élément peut être inutile.
# Si vous l'incluez, placez-le plutôt vers la fin du prompt qu'au début.
OUTPUT_FORMATTING = ""

##### Élément 10 : pré-remplissage de la réponse de Claude (facultatif)
# Emplacement pour commencer la réponse de Claude par quelques mots afin d'orienter son comportement ou sa réponse.
# Pour pré-remplir la réponse de Claude, placez ce texte dans le rôle `assistant` de l'appel API.
# Selon la tâche, cet élément peut être inutile.
PREFILL = ""



######################################## ASSEMBLAGE DES ÉLÉMENTS ########################################

PROMPT = ""

if TASK_CONTEXT:
    PROMPT += f"""{TASK_CONTEXT}"""

if TONE_CONTEXT:
    PROMPT += f"""\n\n{TONE_CONTEXT}"""

if INPUT_DATA:
    PROMPT += f"""\n\n{INPUT_DATA}"""

if EXAMPLES:
    PROMPT += f"""\n\n{EXAMPLES}"""

if TASK_DESCRIPTION:
    PROMPT += f"""\n\n{TASK_DESCRIPTION}"""

if IMMEDIATE_TASK:
    PROMPT += f"""\n\n{IMMEDIATE_TASK}"""

if PRECOGNITION:
    PROMPT += f"""\n\n{PRECOGNITION}"""

if OUTPUT_FORMATTING:
    PROMPT += f"""\n\n{OUTPUT_FORMATTING}"""

# Afficher le prompt complet
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN")
print(PROMPT)
print("\nASSISTANT TURN")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))

❓ Pour voir une solution possible, exécutez la cellule ci-dessous.

In [ ]:
from hints import exercise_9_1_solution; print(exercise_9_1_solution)

### Exercice 9.2 - Codebot
Dans cet exercice, vous rédigez le prompt d'un **assistant de programmation pédagogique qui lit du code et propose des corrections guidées quand c'est pertinent**. Remplissez les champs des éléments du prompt avec un contenu conforme à leur description et aux exemples de prompts complexes vus plus haut. Une fois les éléments voulus remplis, exécutez la cellule pour voir le prompt assemblé et la réponse de Claude.

Nous vous conseillons de lire le contenu de la variable (ici `{CODE}`) pour comprendre sur quoi Claude doit travailler. Veillez à faire référence directement à `{CODE}` quelque part dans votre prompt (avec la syntaxe f-string, comme dans les autres exemples) pour que le contenu réel de la variable y soit inséré.

In [ ]:
######################################## VARIABLES D'ENTRÉE ########################################

# Variable d'entrée : le code que Claude doit lire et aider l'utilisateur à corriger
CODE = """
# Fonction qui affiche les inverses multiplicatifs
def print_multiplicative_inverses(x, n):
  for i in range(n):
    print(x / i) 
"""



######################################## ÉLÉMENTS DU PROMPT ########################################

##### Élément 1 : rôle `user`
# Votre appel à l'API Messages doit toujours commencer par un rôle `user` dans le tableau messages.
# La fonction get_completion() définie plus haut le fait automatiquement pour vous.

##### Élément 2 : contexte de la tâche
# Donnez à Claude le contexte du rôle à endosser, ou des objectifs et de la tâche générale que le prompt doit lui faire accomplir.
# Il vaut mieux placer le contexte au début du prompt.
TASK_CONTEXT = ""

##### Élément 3 : ton
# Si c'est important pour l'échange, indiquez à Claude le ton à adopter.
# Selon la tâche, cet élément peut être inutile.
TONE_CONTEXT = ""

##### Élément 4 : description détaillée de la tâche et règles
# Détaillez les tâches précises que Claude doit réaliser et les règles qu'il doit respecter.
# C'est aussi ici que vous laissez à Claude une « porte de sortie » s'il n'a pas de réponse ou ne sait pas.
# Idéalement, faites relire cette description et ces règles à quelqu'un pour vérifier qu'elles sont logiques et que les termes ambigus sont bien définis.
TASK_DESCRIPTION = ""

##### Élément 5 : exemples
# Fournissez à Claude au moins un exemple de réponse idéale à imiter, entre balises XML <example></example>. Vous pouvez en fournir plusieurs.
# Si vous fournissez plusieurs exemples, précisez à Claude ce que chacun illustre et placez chaque exemple dans ses propres balises XML.
# Les exemples sont sans doute l'outil le plus efficace pour obtenir de Claude le comportement voulu.
# Donnez des exemples des cas limites courants. Si votre prompt utilise un brouillon (scratchpad), montrez à quoi il doit ressembler.
# En général, plus il y a d'exemples, mieux c'est.
EXAMPLES = ""

##### Élément 6 : données d'entrée à traiter
# Si Claude doit traiter des données dans le prompt, placez-les ici entre balises XML appropriées.
# Vous pouvez inclure plusieurs données, chacune dans ses propres balises XML.
# Selon la tâche, cet élément peut être inutile. Son emplacement est libre.
INPUT_DATA = ""

##### Élément 7 : description ou demande de la tâche immédiate #####
# « Rappelez » à Claude, ou dites-lui précisément, ce qu'il doit faire immédiatement pour accomplir la tâche.
# C'est aussi ici que vous placez des variables supplémentaires, comme la question de l'utilisateur.
# Répéter la tâche immédiate ne nuit généralement pas. Faites-le plutôt vers la fin d'un long prompt.
# Les résultats seront meilleurs qu'en la plaçant au début.
# Il est aussi recommandé de placer la question de l'utilisateur vers la fin du prompt.
IMMEDIATE_TASK = ""

##### Élément 8 : réflexion préalable (raisonnement pas à pas)
# Pour les tâches en plusieurs étapes, demandez à Claude de réfléchir étape par étape avant de répondre.
# Il faut parfois écrire « Avant de donner ta réponse... » pour que Claude le fasse bien en premier.
# Pas nécessaire pour tous les prompts. Si vous l'incluez, placez-le vers la fin d'un long prompt, juste après la demande de tâche immédiate.
PRECOGNITION = ""

##### Élément 9 : format de sortie
# Si la réponse doit suivre un format précis, décrivez clairement ce format à Claude.
# Selon la tâche, cet élément peut être inutile.
# Si vous l'incluez, placez-le plutôt vers la fin du prompt qu'au début.
OUTPUT_FORMATTING = ""

##### Élément 10 : pré-remplissage de la réponse de Claude (facultatif)
# Emplacement pour commencer la réponse de Claude par quelques mots afin d'orienter son comportement ou sa réponse.
# Pour pré-remplir la réponse de Claude, placez ce texte dans le rôle `assistant` de l'appel API.
# Selon la tâche, cet élément peut être inutile.
PREFILL = ""



######################################## ASSEMBLAGE DES ÉLÉMENTS ########################################

PROMPT = ""

if TASK_CONTEXT:
    PROMPT += f"""{TASK_CONTEXT}"""

if TONE_CONTEXT:
    PROMPT += f"""\n\n{TONE_CONTEXT}"""

if TASK_DESCRIPTION:
    PROMPT += f"""\n\n{TASK_DESCRIPTION}"""

if EXAMPLES:
    PROMPT += f"""\n\n{EXAMPLES}"""

if INPUT_DATA:
    PROMPT += f"""\n\n{INPUT_DATA}"""

if IMMEDIATE_TASK:
    PROMPT += f"""\n\n{IMMEDIATE_TASK}"""

if PRECOGNITION:
    PROMPT += f"""\n\n{PRECOGNITION}"""

if OUTPUT_FORMATTING:
    PROMPT += f"""\n\n{OUTPUT_FORMATTING}"""

# Afficher le prompt complet
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN")
print(PROMPT)
print("\nASSISTANT TURN")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))

❓ Pour voir une solution possible, exécutez la cellule ci-dessous.

In [ ]:
from hints import exercise_9_2_solution; print(exercise_9_2_solution)

---

## Félicitations et prochaines étapes

Si vous avez fait tous les exercices, **vous faites désormais partie des 0,1 % de meilleurs « chuchoteurs de LLM »**. L'élite !

Les techniques apprises, du raisonnement pas à pas à l'attribution de rôles, en passant par les exemples et la rédaction claire en général, peuvent être **combinées, remaniées et adaptées de très nombreuses façons**.

Le prompt engineering est une discipline très récente : gardez l'esprit ouvert. C'est peut-être vous qui découvrirez la prochaine grande astuce.

Pour trouver **d'autres exemples de bons prompts** :
- Inspirez-vous des prompts prêts pour la production de notre [cookbook](https://anthropic.com/cookbook).
- Lisez notre [guide du prompt engineering](https://docs.anthropic.com/claude/docs/prompt-engineering).
- Parcourez notre [bibliothèque de prompts](https://anthropic.com/prompts).
- Essayez notre [métaprompt expérimental](https://docs.anthropic.com/claude/docs/helper-metaprompt-experimental) pour que Claude rédige des modèles de prompt à votre place.
- Posez vos questions sur notre [serveur Discord](https://anthropic.com/discord).
- Découvrez les [paramètres de l'API Anthropic](https://docs.anthropic.com/claude/reference/complete_post), comme la température et `max_tokens`.
- Si vous aimez la théorie, lisez quelques [articles de recherche](https://www.promptingguide.ai/papers) sur le prompt engineering.
- Entraînez-vous à construire des prompts pour faire faire à Claude quelque chose qui vous intéresse.

Pour découvrir des techniques vraiment avancées qui dépassent le cadre de ce tutoriel, passez à l'annexe ! Mais exécutez d'abord la cellule ci-dessous.

In [ ]:
# Prompt
PROMPT = "Write an ode to a fabulous student who has just completed a course on prompt engineering, in the form of a sonnet."

# Afficher la réponse de Claude
print(get_completion(PROMPT))

---

## Espace de test

Cet espace vous permet d'expérimenter librement avec les exemples de prompts de la leçon et de les modifier pour voir l'effet sur les réponses de Claude.

In [ ]:
######################################## VARIABLES D'ENTRÉE ########################################

# Première variable d'entrée : l'historique de conversation (peut aussi être passé sous forme de messages `user` et `assistant` précédents dans l'appel API)
HISTORY = """Customer: Give me two possible careers for sociology majors.

Joe: Here are two potential careers for sociology majors:

- Social worker - Sociology provides a strong foundation for understanding human behavior and social systems. With additional training or certification, a sociology degree can qualify graduates for roles as social workers, case managers, counselors, and community organizers helping individuals and groups.

- Human resources specialist - An understanding of group dynamics and organizational behavior from sociology is applicable to careers in human resources. Graduates may find roles in recruiting, employee relations, training and development, diversity and inclusion, and other HR functions. The focus on social structures and institutions also supports related careers in public policy, nonprofit management, and education."""

# Deuxième variable d'entrée : la question de l'utilisateur
QUESTION = "Which of the two careers requires more than a Bachelor's degree?"



######################################## ÉLÉMENTS DU PROMPT ########################################

##### Élément 1 : rôle `user`
# Votre appel à l'API Messages doit toujours commencer par un rôle `user` dans le tableau messages.
# La fonction get_completion() définie plus haut le fait automatiquement pour vous.

##### Élément 2 : contexte de la tâche
# Donnez à Claude le contexte du rôle à endosser, ou des objectifs et de la tâche générale que le prompt doit lui faire accomplir.
# Il vaut mieux placer le contexte au début du prompt.
TASK_CONTEXT = "You will be acting as an AI career coach named Joe created by the company AdAstra Careers. Your goal is to give career advice to users. You will be replying to users who are on the AdAstra site and who will be confused if you don't respond in the character of Joe."

##### Élément 3 : ton
# Si c'est important pour l'échange, indiquez à Claude le ton à adopter.
# Selon la tâche, cet élément peut être inutile.
TONE_CONTEXT = "You should maintain a friendly customer service tone."

##### Élément 4 : description détaillée de la tâche et règles
# Détaillez les tâches précises que Claude doit réaliser et les règles qu'il doit respecter.
# C'est aussi ici que vous laissez à Claude une « porte de sortie » s'il n'a pas de réponse ou ne sait pas.
# Idéalement, faites relire cette description et ces règles à quelqu'un pour vérifier qu'elles sont logiques et que les termes ambigus sont bien définis.
TASK_DESCRIPTION = """Here are some important rules for the interaction:
- Always stay in character, as Joe, an AI from AdAstra Careers
- If you are unsure how to respond, say \"Sorry, I didn't understand that. Could you rephrase your question?\"
- If someone asks something irrelevant, say, \"Sorry, I am Joe and I give career advice. Do you have a career question today I can help you with?\""""

##### Élément 5 : exemples
# Fournissez à Claude au moins un exemple de réponse idéale à imiter, entre balises XML <example></example>. Vous pouvez en fournir plusieurs.
# Si vous fournissez plusieurs exemples, précisez à Claude ce que chacun illustre et placez chaque exemple dans ses propres balises XML.
# Les exemples sont sans doute l'outil le plus efficace pour obtenir de Claude le comportement voulu.
# Donnez des exemples des cas limites courants. Si votre prompt utilise un brouillon (scratchpad), montrez à quoi il doit ressembler.
# En général, plus il y a d'exemples, mieux c'est.
EXAMPLES = """Here is an example of how to respond in a standard interaction:
<example>
Customer: Hi, how were you created and what do you do?
Joe: Hello! My name is Joe, and I was created by AdAstra Careers to give career advice. What can I help you with today?
</example>"""

##### Élément 6 : données d'entrée à traiter
# Si Claude doit traiter des données dans le prompt, placez-les ici entre balises XML appropriées.
# Vous pouvez inclure plusieurs données, chacune dans ses propres balises XML.
# Selon la tâche, cet élément peut être inutile. Son emplacement est libre.
INPUT_DATA = f"""Here is the conversational history (between the user and you) prior to the question. It could be empty if there is no history:
<history>
{HISTORY}
</history>

Here is the user's question:
<question>
{QUESTION}
</question>"""

##### Élément 7 : description ou demande de la tâche immédiate #####
# « Rappelez » à Claude, ou dites-lui précisément, ce qu'il doit faire immédiatement pour accomplir la tâche.
# C'est aussi ici que vous placez des variables supplémentaires, comme la question de l'utilisateur.
# Répéter la tâche immédiate ne nuit généralement pas. Faites-le plutôt vers la fin d'un long prompt.
# Les résultats seront meilleurs qu'en la plaçant au début.
# Il est aussi recommandé de placer la question de l'utilisateur vers la fin du prompt.
IMMEDIATE_TASK = "How do you respond to the user's question?"

##### Élément 8 : réflexion préalable (raisonnement pas à pas)
# Pour les tâches en plusieurs étapes, demandez à Claude de réfléchir étape par étape avant de répondre.
# Il faut parfois écrire « Avant de donner ta réponse... » pour que Claude le fasse bien en premier.
# Pas nécessaire pour tous les prompts. Si vous l'incluez, placez-le vers la fin d'un long prompt, juste après la demande de tâche immédiate.
PRECOGNITION = "Think about your answer first before you respond."

##### Élément 9 : format de sortie
# Si la réponse doit suivre un format précis, décrivez clairement ce format à Claude.
# Selon la tâche, cet élément peut être inutile.
# Si vous l'incluez, placez-le plutôt vers la fin du prompt qu'au début.
OUTPUT_FORMATTING = "Put your response in <response></response> tags."

##### Élément 10 : pré-remplissage de la réponse de Claude (facultatif)
# Emplacement pour commencer la réponse de Claude par quelques mots afin d'orienter son comportement ou sa réponse.
# Pour pré-remplir la réponse de Claude, placez ce texte dans le rôle `assistant` de l'appel API.
# Selon la tâche, cet élément peut être inutile.
PREFILL = "[Joe] <response>"



######################################## ASSEMBLAGE DES ÉLÉMENTS ########################################

PROMPT = ""

if TASK_CONTEXT:
    PROMPT += f"""{TASK_CONTEXT}"""

if TONE_CONTEXT:
    PROMPT += f"""\n\n{TONE_CONTEXT}"""

if TASK_DESCRIPTION:
    PROMPT += f"""\n\n{TASK_DESCRIPTION}"""

if EXAMPLES:
    PROMPT += f"""\n\n{EXAMPLES}"""

if INPUT_DATA:
    PROMPT += f"""\n\n{INPUT_DATA}"""

if IMMEDIATE_TASK:
    PROMPT += f"""\n\n{IMMEDIATE_TASK}"""

if PRECOGNITION:
    PROMPT += f"""\n\n{PRECOGNITION}"""

if OUTPUT_FORMATTING:
    PROMPT += f"""\n\n{OUTPUT_FORMATTING}"""

# Afficher le prompt complet
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN")
print(PROMPT)
print("\nASSISTANT TURN")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))

In [ ]:
######################################## VARIABLES D'ENTRÉE ########################################

# Première variable d'entrée : le document juridique
LEGAL_RESEARCH = """<search_results>
<search_result id=1>
The animal health industry became caught up in a number of patent and trademark lawsuits during the past year. In 1994, Barclay Slocum obtained patents for the tibial plateau leveling osteotomy procedure, which is used in the treatment of dogs with cranial cruciate ligament rupture, and for the devices used in the procedure. During 2006, Slocum Enterprises filed a patent infringement suit against New Generation Devices, arguing that the Unity Cruciate Plate manufactured by New Generation infringed on the patent for the Slocum TPLO plate. However, the court never reached a decision on the issue of patent infringement, ruling that it did not have jurisdiction on the basis of the small number of plates sold in the state in which the case was filed and the information provided on a Web site maintained by Slocum Enterprises. Other patent battles waged during 2006 concerned the use of laser technology for onychectomy in cats, pet identification chips, pig vaccines, and pet “deshedding” tools.
</search_result>
<search_result id=2>
In Canada, the British Columbia Veterinary Medical Association brought suit against a nonveterinarian, claiming that he engaged in cutting or otherwise removing hooks from horses' teeth and floating horses' teeth with power and manual tools, provided advice and diagnoses in return for a fee, and held himself out as being qualified and willing to provide treatment with respect to these activities. The court held that the intention of the legislature in passing the Veterinary Profession Act was the protection of the public and animals and further held that monopolistic statutes serve the purpose of protecting the public. In addition, the court concluded that dentistry, at its core, relates to the health of the teeth and gums; is distinct from cosmetic and other types of care of animals; and, therefore, falls under the definition of the practice of veterinary medicine. The nonveterinarian was enjoined from providing services without a veterinarian supervising the procedures.
</search_result>
<search_result id=3>
The aftermath of Hurricane Katrina, which hit the Gulf Coast of the United States during 2005, spurred changes to the way animals are treated during natural disasters. In 2006, Hawaii, Louisiana, and New Hampshire all enacted laws that address issues regarding the care of animals during disasters, such as providing shelters for pets and allowing service animals to be kept with the people they serve. In addition, Congress passed, and the President signed, the Pet Evacuation and Transportation Standards Act during 2006, which requires state and local emergency preparedness authorities to include in their evacuation plans information on how they will accommodate household pets and service animals in case of a disaster. California passed a law that will require its Office of Emergency Services, Department of Agriculture, and other agencies involved with disaster response preparation to develop a plan for the needs of service animals, livestock, equids, and household pets in the event of a disaster or major emergency.
</search_result>
</search_results>"""

# Deuxième variable d'entrée : la question de l'utilisateur
QUESTION = "Are there any laws about what to do with pets during a hurricane?"



######################################## ÉLÉMENTS DU PROMPT ########################################

##### Élément 1 : rôle `user`
# Votre appel à l'API Messages doit toujours commencer par un rôle `user` dans le tableau messages.
# La fonction get_completion() définie plus haut le fait automatiquement pour vous.

##### Élément 2 : contexte de la tâche
# Donnez à Claude le contexte du rôle à endosser, ou des objectifs et de la tâche générale que le prompt doit lui faire accomplir.
# Il vaut mieux placer le contexte au début du prompt.
TASK_CONTEXT = "You are an expert lawyer."

##### Élément 3 : ton
# Si c'est important pour l'échange, indiquez à Claude le ton à adopter.
# Selon la tâche, cet élément peut être inutile.
TONE_CONTEXT = ""

##### Élément 4 : données d'entrée à traiter
# Si Claude doit traiter des données dans le prompt, placez-les ici entre balises XML appropriées.
# Vous pouvez inclure plusieurs données, chacune dans ses propres balises XML.
# Selon la tâche, cet élément peut être inutile. Son emplacement est libre.
INPUT_DATA = f"""Here is some research that's been compiled. Use it to answer a legal question from the user.
<legal_research>
{LEGAL_RESEARCH}
</legal_research>"""

##### Élément 5 : exemples
# Fournissez à Claude au moins un exemple de réponse idéale à imiter, entre balises XML <example></example>. Vous pouvez en fournir plusieurs.
# Si vous fournissez plusieurs exemples, précisez à Claude ce que chacun illustre et placez chaque exemple dans ses propres balises XML.
# Les exemples sont sans doute l'outil le plus efficace pour obtenir de Claude le comportement voulu.
# Donnez des exemples des cas limites courants. Si votre prompt utilise un brouillon (scratchpad), montrez à quoi il doit ressembler.
# En général, plus il y a d'exemples, mieux c'est.
EXAMPLES = """When citing the legal research in your answer, please use brackets containing the search index ID, followed by a period. Put these at the end of the sentence that's doing the citing. Examples of proper citation format:

<examples>
<example>
The statute of limitations expires after 10 years for crimes like this. [3].
</example>
<example>
However, the protection does not apply when it has been specifically waived by both parties. [5].
</example>
</examples>"""

##### Élément 6 : description détaillée de la tâche et règles
# Détaillez les tâches précises que Claude doit réaliser et les règles qu'il doit respecter.
# C'est aussi ici que vous laissez à Claude une « porte de sortie » s'il n'a pas de réponse ou ne sait pas.
# Idéalement, faites relire cette description et ces règles à quelqu'un pour vérifier qu'elles sont logiques et que les termes ambigus sont bien définis.
TASK_DESCRIPTION = """Write a clear, concise answer to this question:

<question>
{QUESTION}
</question>

It should be no more than a couple of paragraphs. If possible, it should conclude with a single sentence directly answering the user's question. However, if there is not sufficient information in the compiled research to produce such an answer, you may demur and write "Sorry, I do not have sufficient information at hand to answer this question."."""

##### Élément 7 : description ou demande de la tâche immédiate #####
# « Rappelez » à Claude, ou dites-lui précisément, ce qu'il doit faire immédiatement pour accomplir la tâche.
# C'est aussi ici que vous placez des variables supplémentaires, comme la question de l'utilisateur.
# Répéter la tâche immédiate ne nuit généralement pas. Faites-le plutôt vers la fin d'un long prompt.
# Les résultats seront meilleurs qu'en la plaçant au début.
# Il est aussi recommandé de placer la question de l'utilisateur vers la fin du prompt.
IMMEDIATE_TASK = ""

##### Élément 8 : réflexion préalable (raisonnement pas à pas)
# Pour les tâches en plusieurs étapes, demandez à Claude de réfléchir étape par étape avant de répondre.
# Il faut parfois écrire « Avant de donner ta réponse... » pour que Claude le fasse bien en premier.
# Pas nécessaire pour tous les prompts. Si vous l'incluez, placez-le vers la fin d'un long prompt, juste après la demande de tâche immédiate.
PRECOGNITION = "Before you answer, pull out the most relevant quotes from the research in <relevant_quotes> tags."

##### Élément 9 : format de sortie
# Si la réponse doit suivre un format précis, décrivez clairement ce format à Claude.
# Selon la tâche, cet élément peut être inutile.
# Si vous l'incluez, placez-le plutôt vers la fin du prompt qu'au début.
OUTPUT_FORMATTING = "Put your two-paragraph response in <answer> tags."

##### Élément 10 : pré-remplissage de la réponse de Claude (facultatif)
# Emplacement pour commencer la réponse de Claude par quelques mots afin d'orienter son comportement ou sa réponse.
# Pour pré-remplir la réponse de Claude, placez ce texte dans le rôle `assistant` de l'appel API.
# Selon la tâche, cet élément peut être inutile.
PREFILL = "<relevant_quotes>"



######################################## ASSEMBLAGE DES ÉLÉMENTS ########################################

PROMPT = ""

if TASK_CONTEXT:
    PROMPT += f"""{TASK_CONTEXT}"""

if TONE_CONTEXT:
    PROMPT += f"""\n\n{TONE_CONTEXT}"""

if INPUT_DATA:
    PROMPT += f"""\n\n{INPUT_DATA}"""

if EXAMPLES:
    PROMPT += f"""\n\n{EXAMPLES}"""

if TASK_DESCRIPTION:
    PROMPT += f"""\n\n{TASK_DESCRIPTION}"""

if IMMEDIATE_TASK:
    PROMPT += f"""\n\n{IMMEDIATE_TASK}"""

if PRECOGNITION:
    PROMPT += f"""\n\n{PRECOGNITION}"""

if OUTPUT_FORMATTING:
    PROMPT += f"""\n\n{OUTPUT_FORMATTING}"""

# Afficher le prompt complet
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN")
print(PROMPT)
print("\nASSISTANT TURN")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))